<a href="https://colab.research.google.com/github/szymonszwedzinskiii/DataTemplates/blob/main/LogisticRegresionVsDecisionTree.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Imports:

In [169]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from collections import Counter

Logistic regresion

In [144]:
df = pd.read_csv('/content/Crop_recommendation.csv')

In [145]:
X = df.iloc[:,:-1].values
y = df.iloc[:,-1].values
y = LabelEncoder().fit_transform(y)
X_train,X_test, y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)
w = np.random.randn(X_train.shape[1], len(np.unique(y_train))) * 0.01
b = np.zeros(len(np.unique(y_train)))


In [146]:
def softmax(logits):
  return np.exp(logits) / np.sum(np.exp(logits), axis=0)

In [147]:
def softmax(logits):
    logits = np.atleast_2d(logits)  # gwarantuje 2D: (m, K)
    exp = np.exp(logits - np.max(logits, axis=1, keepdims=True))  # stabilizacja
    return exp / np.sum(exp, axis=1, keepdims=True)

In [148]:
def model(X,w,b):
  return softmax(np.dot(X,w) + b)

In [149]:
def compute_cost(X,y,w,b):
  m = X.shape[0]
  cost = 0
  pred = model(X,w,b)

  for i in range(m):
    cost += -np.log(pred[i,y[i]])
  return cost/m

In [150]:
pred = model(X_train,w,b)
print(pred)

[[0.0023244  0.0028841  0.0910326  ... 0.00619216 0.08573551 0.01928742]
 [0.00688013 0.00368327 0.18286657 ... 0.00217181 0.02015903 0.03408472]
 [0.01144131 0.00900063 0.18996562 ... 0.00318402 0.02706564 0.03634437]
 ...
 [0.0041898  0.00876363 0.10228356 ... 0.0091614  0.07666233 0.02339142]
 [0.0003296  0.01780033 0.09251662 ... 0.00463495 0.01481283 0.00182296]
 [0.00973944 0.00559309 0.17992123 ... 0.00291157 0.01979614 0.0370474 ]]


In [151]:
cost = compute_cost(X_train,y_train,w,b)

print(cost)

4.283477609161649


In [152]:
def gradient_descent(X,y,w,b,learning_rate,iterations):
  m = X.shape[0]
  for i in range(iterations):
    cost_history = []
    w_history = []
    b_history = []
    pred = model(X,w,b)
    cost = compute_cost(X,y,w,b)
    cost_history.append(cost)
    w_history.append(w)
    b_history.append(b)
    y_onehot = np.zeros_like(pred)
    y_onehot[np.arange(len(y)), y] = 1
    dw = (1/m) * np.dot(X.T,(pred - y_onehot))
    db = (1/m) * np.sum(pred - y_onehot)

    w -= learning_rate * dw
    b -= learning_rate * db
  return cost_history,w_history,b_history
cost_history, w_history, b_history = gradient_descent(X_train,y_train,w,b,0.01,1000)

cost = compute_cost(X_test,y_test,w_history[-1],b_history[-1])
print(f'Cost on training data = {cost_history[-1]} and cost on test data = {cost}')


Cost on training data = 0.5917062232706777 and cost on test data = 0.6471995925559854


Decission tree

In [161]:
df = pd.read_csv('/content/Crop_recommendation.csv')
X = df.iloc[:,:-1].values
y = df.iloc[:,-1].values
y = LabelEncoder().fit_transform(y)
X_train,X_test, y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)
w = np.random.randn(X_train.shape[1], len(np.unique(y_train))) * 0.01
b = np.zeros(len(np.unique(y_train)))

In [154]:
def gini_index(groups, classes):
    total_instances = sum(len(group) for group in groups)
    gini = 0.0
    for group in groups:
        size = len(group)
        if size == 0:
            continue
        score = 0.0
        labels = [row[-1] for row in group]
        for c in classes:
            p = labels.count(c) / size
            score += p * p
        gini += (1.0 - score) * (size / total_instances)
    return gini

In [155]:
def test_split(index, value, dataset):
    left, right = [], []
    for row in dataset:
        if row[index] <= value:
            left.append(row)
        else:
            right.append(row)
    return left, right

In [156]:
def get_split(dataset):
    class_values = list(set(row[-1] for row in dataset))
    best_index, best_value, best_score, best_groups = None, None, float('inf'), None
    for index in range(len(dataset[0]) - 1):
        for row in dataset:
            groups = test_split(index, row[index], dataset)
            gini = gini_index(groups, class_values)
            if gini < best_score:
                best_index, best_value, best_score, best_groups = index, row[index], gini, groups
    return {"index": best_index, "value": best_value, "groups": best_groups}

In [167]:
def to_terminal(group):
    outcomes = [row[-1] for row in group]
    return Counter(outcomes).most_common(1)[0][0]


In [158]:
def split(node, max_depth, min_size, depth):
    left, right = node["groups"]
    del(node["groups"])

    if not left or not right:
        node["left"] = node["right"] = to_terminal(left + right)
        return

    if depth >= max_depth:
        node["left"], node["right"] = to_terminal(left), to_terminal(right)
        return

    if len(left) <= min_size:
        node["left"] = to_terminal(left)
    else:
        node["left"] = get_split(left)
        split(node["left"], max_depth, min_size, depth + 1)

    if len(right) <= min_size:
        node["right"] = to_terminal(right)
    else:
        node["right"] = get_split(right)
        split(node["right"], max_depth, min_size, depth + 1)


In [159]:
def build_tree(train, max_depth, min_size):
    root = get_split(train)
    split(root, max_depth, min_size, 1)
    return root

In [160]:
def predict(node, row):
    if row[node['index']] <= node['value']:
        if isinstance(node['left'], dict):
            return predict(node['left'], row)
        else:
            return node['left']
    else:
        if isinstance(node['right'], dict):
            return predict(node['right'], row)
        else:
            return node['right']

In [178]:
dataset = df.values.tolist()
tree = build_tree(dataset[:1000], 5, 10)
correct =0
for row in df.values:
    prediction = predict(tree, row[:-1])
    #print(f'Actual: {row[-1]}, Predicted: {prediction}')
    if row[-1] == prediction:
        correct += 1
accuracy = correct / len(df)
print(f'Accuracy: {round(accuracy,3)}%')

Accuracy: 0.273%
